# NB01 — Extração e Consolidação

Download do Kaggle, limpeza, tipagem, tratamento de nulos, padronização de categorias.

**Output:** `../data/interm/base_consolidada.parquet`

### Diretórios e download do dataset

In [1]:
import kaggle
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")
INTERM_DIR = Path("../data/interm")
CSV_PATH = RAW_DIR / "WA_Fn-UseC_-Telco-Customer-Churn.csv"

for pasta in (RAW_DIR, INTERM_DIR, Path("../data/processed"), Path("../models")):
    pasta.mkdir(parents=True, exist_ok=True)

if not CSV_PATH.exists():
    kaggle.api.authenticate()
    kaggle.api.dataset_download_files(
        "blastchar/telco-customer-churn",
        path=str(RAW_DIR),
        unzip=True,
    )
    print("Download concluído.")
else:
    print(f"CSV já presente: {CSV_PATH}")

CSV já presente: ..\data\raw\WA_Fn-UseC_-Telco-Customer-Churn.csv


### Carregamento e inspeção inicial

In [2]:
df = pd.read_csv(CSV_PATH)
print(f"Shape original: {df.shape[0]:,} linhas x {df.shape[1]} colunas")

print("\n--- Tipos ---")
print(df.dtypes)

print("\n--- Amostra ---")
display(df.head())

ausentes = df.isnull().sum()
print("\n--- Ausentes ---")
print(ausentes[ausentes > 0] if ausentes.any() else "Nenhum")

print("\n--- Descritiva ---")
display(df.describe(include="all"))

Shape original: 7,043 linhas x 21 colunas

--- Tipos ---
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges         object
Churn                object
dtype: object

--- Amostra ---


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes



--- Ausentes ---
Nenhum

--- Descritiva ---


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
count,7043,7043,7043.000000,7043,7043,7043.000000,7043,7043,7043,7043,...,7043,7043,7043,7043,7043,7043,7043,7043.000000,7043,7043
unique,7043,2,NaN,2,2,NaN,2,3,3,3,...,3,3,3,3,3,2,4,NaN,6531,2
top,7590-VHVEG,Male,NaN,No,No,NaN,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,NaN,20.2,No
freq,1,3555,NaN,3641,4933,NaN,6361,3390,3096,3498,...,3095,3473,2810,2785,3875,4171,2365,NaN,11,5174
mean,NaN,NaN,0.162147,NaN,NaN,32.371149,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,64.761692,NaN,NaN
std,NaN,NaN,0.368612,NaN,NaN,24.559481,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,30.090047,NaN,NaN
min,NaN,NaN,0.000000,NaN,NaN,0.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,18.250000,NaN,NaN
25%,NaN,NaN,0.000000,NaN,NaN,9.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,35.500000,NaN,NaN
50%,NaN,NaN,0.000000,NaN,NaN,29.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,70.350000,NaN,NaN
75%,NaN,NaN,0.000000,NaN,NaN,55.000000,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,89.850000,NaN,NaN


### Tipagem, nulos e padronização de categorias

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
print(f"Nulos em TotalCharges após conversão: {df['TotalCharges'].isnull().sum()}")
display(df[df["TotalCharges"].isnull()][["tenure", "MonthlyCharges", "TotalCharges"]])

# Clientes com tenure 0: TotalCharges ausente → 0.0
df["TotalCharges"] = df["TotalCharges"].fillna(0.0)

df["Churn"] = (df["Churn"] == "Yes").astype(int)

servicos = [
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "MultipleLines",
]
for col in servicos:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

Nulos em TotalCharges após conversão: 11


,tenure,MonthlyCharges,TotalCharges
488,0,52.55,NaN
753,0,20.25,NaN
936,0,80.85,NaN
1082,0,25.75,NaN
1340,0,56.05,NaN
3331,0,19.85,NaN
3826,0,25.35,NaN
4380,0,20.00,NaN
5218,0,19.70,NaN
6670,0,73.35,NaN


### Validação e exportação da base consolidada

In [4]:
churn_dist = df["Churn"].value_counts().sort_index()
churn_pct = df["Churn"].value_counts(normalize=True).sort_index() * 100
print("Distribuição Churn:")
for rotulo, nome in [(0, "Não churn"), (1, "Churn")]:
    print(f"  {nome} ({rotulo}): {churn_dist[rotulo]:,} ({churn_pct[rotulo]:.1f}%)")

assert df["customerID"].nunique() == len(df)
assert df["TotalCharges"].isnull().sum() == 0
assert df["Churn"].isin([0, 1]).all()

print(f"\nShape final: {df.shape[0]:,} x {df.shape[1]}")

saida = INTERM_DIR / "base_consolidada.parquet"
df.to_parquet(saida, engine="pyarrow", index=False)
print(f"Salvo: {saida}")

Distribuição Churn:
  Não churn (0): 5,174 (73.5%)
  Churn (1): 1,869 (26.5%)

Shape final: 7,043 x 21
Salvo: ..\data\interm\base_consolidada.parquet
